# 🌐 Web Scraper Exploration

This notebook demonstrates the web scraping functionality of our college AI pipeline.

**Topics Covered:**
- Initializing the WebScraper
- Testing single URL extraction
- Template similarity comparison
- Quality checking and filtering
- Duplicate detection analysis

## 1. Setup and Imports

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Import our modules
from src.scraper import WebScraper
from src.config import TEMPLATE_URL, CONTENT_FILTER

# Import standard libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Configure display settings
pd.set_option('display.max_colwidth', 100)
sns.set_style('whitegrid')

print("✓ All imports successful!")
print(f"Project root: {project_root}")

## 2. Initialize WebScraper

Let's create a WebScraper instance with our template URL for duplicate detection.

In [ ]:
# Initialize scraper
scraper = WebScraper()

print(f"Template URL: {scraper.template_url}")
print(f"Template loaded: {scraper.template_text is not None}")
if scraper.template_text:
    print(f"Template length: {len(scraper.template_text)} characters")
    print(f"\nFirst 200 chars:\n{scraper.template_text[:200]}...")

## 3. Test Single URL Scraping

Let's scrape a single page and examine the extracted content.

In [ ]:
# Test URL
test_url = "https://sdit.ac.in/about/"

# Scrape the page
result = scraper.scrape(test_url)

if result:
    print("✓ Scraping successful!\n")
    print(f"Title: {result['title']}")
    print(f"URL: {result['url']}")
    print(f"Page name: {result['page_name']}")
    print(f"\nMetadata:")
    for key, value in result['metadata'].items():
        print(f"  {key}: {value}")
    print(f"\nExtracted text ({len(result['text'])} chars):")
    print(result['text'][:500] + "...")
else:
    print("✗ Scraping failed or content rejected")

## 4. Template Similarity Analysis

Compare different pages against the template to understand duplicate detection.

In [ ]:
# URLs to compare
test_urls = [
    "https://sdit.ac.in/about/",
    "https://sdit.ac.in/departments/cse/",
    "https://sdit.ac.in/about-the-placement/",
    "https://sdit.ac.in/karthik-b-2/",  # Student profile (should be similar to template)
]

# Analyze similarity
results = []
for url in test_urls:
    html = scraper.download(url)
    if html:
        text, metadata = scraper.extract_content(html, url)
        similarity = scraper.compare_similarity(text, scraper.template_text) if scraper.template_text else 0
        is_dup = scraper.is_duplicate(text)
        
        results.append({
            'URL': url.split('/')[-2] if url.split('/')[-2] else 'index',
            'Similarity': similarity,
            'Is Duplicate': '✓ Yes' if is_dup else '✗ No',
            'Text Length': len(text),
            'Word Count': metadata['word_count']
        })

# Display results
df = pd.DataFrame(results)
display(df)

# Visualize
plt.figure(figsize=(10, 5))
plt.barh(df['URL'], df['Similarity'], color=['red' if x >= 0.85 else 'green' for x in df['Similarity']])
plt.axvline(x=0.85, color='orange', linestyle='--', label='Duplicate Threshold (0.85)')
plt.xlabel('Similarity Score')
plt.title('Template Similarity Analysis')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Quality Filtering Analysis

Test the quality filters with different content samples.

In [ ]:
# Display current quality thresholds
print("Quality Filter Thresholds:")
print(f"  Minimum text length: {CONTENT_FILTER['min_text_length']} characters")
print(f"  Minimum paragraphs: {CONTENT_FILTER['min_paragraphs']}")
print(f"  Minimum sentences: {CONTENT_FILTER['min_sentences']}")
print()

# Test with actual pages
quality_results = []
for url in test_urls:
    html = scraper.download(url)
    if html:
        text, metadata = scraper.extract_content(html, url)
        passes_quality = scraper.is_quality_content(text, metadata)
        
        quality_results.append({
            'URL': url.split('/')[-2] if url.split('/')[-2] else 'index',
            'Passes': '✓' if passes_quality else '✗',
            'Text Length': len(text),
            'Paragraphs': metadata['paragraphs'],
            'Words': metadata['word_count']
        })

df_quality = pd.DataFrame(quality_results)
display(df_quality)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].barh(df_quality['URL'], df_quality['Text Length'])
axes[0].axvline(x=CONTENT_FILTER['min_text_length'], color='red', linestyle='--', label='Threshold')
axes[0].set_xlabel('Text Length')
axes[0].set_title('Text Length Check')
axes[0].legend()

axes[1].barh(df_quality['URL'], df_quality['Paragraphs'])
axes[1].axvline(x=CONTENT_FILTER['min_paragraphs'], color='red', linestyle='--', label='Threshold')
axes[1].set_xlabel('Paragraphs')
axes[1].set_title('Paragraph Count Check')
axes[1].legend()

axes[2].barh(df_quality['URL'], df_quality['Words'])
axes[2].set_xlabel('Word Count')
axes[2].set_title('Word Count')

plt.tight_layout()
plt.show()

## 6. Summary

**Key Takeaways:**
- ✅ WebScraper successfully extracts clean text from pages
- ✅ Template comparison identifies duplicate/boilerplate pages
- ✅ Quality filters ensure meaningful content
- ✅ Ready for integration with classification pipeline